In [12]:
%load_ext autoreload
%autoreload 2

import pyxdf
import numpy as np
from eeg_toolkit import load_config, find_raw_files

cfg = load_config('../configs/eye_eeg_simul.yaml')
xdf_files = find_raw_files(cfg, extension='xdf')
print(f"Found {len(xdf_files)} XDF files")

# Pick the first one to inspect
sample_xdf = xdf_files[0]
print(f"\nInspecting: {sample_xdf.name}")

streams, header = pyxdf.load_xdf(str(sample_xdf))
print(f"\nNumber of streams: {len(streams)}")
print(f"Header: {header}\n")

for i, s in enumerate(streams):
    info = s['info']
    name = info['name'][0] if info['name'] else "Unnamed"
    stype = info['type'][0] if info['type'] else "Unknown"
    n_channels = int(info['channel_count'][0]) if info['channel_count'] else 0
    sfreq_str = info['nominal_srate'][0] if info['nominal_srate'] else "0"
    sfreq = float(sfreq_str)
    n_samples = len(s['time_series']) if s['time_series'] is not None else 0

    print(f"[Stream {i}]")
    print(f"  name:     {name}")
    print(f"  type:     {stype}")
    print(f"  channels: {n_channels}")
    print(f"  sfreq:    {sfreq}")
    print(f"  samples:  {n_samples}")
    if sfreq > 0:
        print(f"  duration: {n_samples / sfreq:.1f}s")
    else:
        print(f"  duration: (asynchronous stream — no fixed rate)")

    # Show markers if marker stream
    if stype == 'Markers' and n_samples > 0:
        ts = s['time_series']
        first_5 = [v[0] if isinstance(v, (list, np.ndarray)) else v for v in ts[:5]]
        last_5 = [v[0] if isinstance(v, (list, np.ndarray)) else v for v in ts[-5:]]
        print(f"  first 5 markers: {first_5}")
        print(f"  last 5 markers:  {last_5}")
        unique_markers = set()
        for v in ts:
            val = v[0] if isinstance(v, (list, np.ndarray)) else v
            unique_markers.add(str(val))
        print(f"  unique values:   {sorted(unique_markers)}")
        # Count occurrences of each marker
        from collections import Counter
        counter = Counter()
        for v in ts:
            val = v[0] if isinstance(v, (list, np.ndarray)) else v
            counter[str(val)] += 1
        print(f"  marker counts:   {dict(sorted(counter.items()))}")

    # EEG channel names + data range
    if stype == 'EEG':
        try:
            channels = info['desc'][0]['channels'][0]['channel']
            ch_names = [ch['label'][0] for ch in channels]
            print(f"  channel names: {ch_names}")
        except (TypeError, KeyError, IndexError):
            print(f"  channel names: (not in metadata)")
        data = s['time_series']
        print(f"  data range:    min={np.min(data):.3e}, max={np.max(data):.3e}")
        print(f"  data shape:    {data.shape}  (samples x channels)")

    print()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Found 38 XDF files

Inspecting: subj10.xdf

Number of streams: 2
Header: {'info': defaultdict(<class 'list'>, {'version': ['1.0'], 'datetime': ['2026-02-13T14:34:15-0500']})}

[Stream 0]
  name:     actiCHamp-16090700
  type:     EEG
  channels: 32
  sfreq:    500.0
  samples:  970757
  duration: 1941.5s
  channel names: ['Fp1', 'Fz', 'F3', 'F7', 'FT9', 'FC5', 'FC1', 'C3', 'T7', 'TP9', 'CP5', 'CP1', 'Pz', 'P3', 'P7', 'O1', 'Oz', 'O2', 'P4', 'P8', 'TP10', 'CP6', 'CP2', 'Cz', 'C4', 'T8', 'FT10', 'FC6', 'FC2', 'F4', 'F8', 'Fp2']
  data range:    min=-3.044e+04, max=1.110e+04
  data shape:    (970757, 32)  (samples x channels)

[Stream 1]
  name:     actiCHampMarkers-16090700
  type:     Markers
  channels: 1
  sfreq:    0.0
  samples:  1680
  duration: (asynchronous stream — no fixed rate)
  first 5 markers: ['50', '20', '51', '34', '52']
  last 5 markers:  ['51', '123', '52', '40', '90']
  unique valu

In [4]:
%load_ext autoreload
%autoreload 2

from eeg_toolkit import load_config, find_subjects, get_excluded_subjects

cfg = load_config('../configs/eye_eeg_simul.yaml')

excluded = get_excluded_subjects(cfg)
print(f"Excluded per config ({len(excluded)}):")
print(sorted(excluded))
print()

all_subjects = find_subjects(cfg, apply_exclusions=False)
print(f"All subject folders on disk ({len(all_subjects)}):")
print(all_subjects)
print()

included = find_subjects(cfg)
print(f"Subjects to process ({len(included)}):")
print(included)

included = find_subjects(cfg, apply_exclusions=False)
print(f"All folders on disk: {len(included)}")

included = find_subjects(cfg)
print(f"Subjects after exclusions: {len(included)}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Excluded per config (12):
['subj1', 'subj11', 'subj12', 'subj13', 'subj15', 'subj16', 'subj2', 'subj20', 'subj28', 'subj42', 'subj7', 'subj9']

All subject folders on disk (1):
['subj10']

Subjects to process (1):
['subj10']
All folders on disk: 1
Subjects after exclusions: 1
